# Explainability Artifacts with MLflow

## Topic Goal

This notebook properly implements the expected **Explainability Artifacts** use case with MLflow.

In production MLOps, we should not only log the model and metrics.  
We should also log explainability and diagnostic artifacts so that reviewers can understand model behavior.

This notebook demonstrates:

1. Load customer churn dataset.
2. Train a classification model.
3. Evaluate model performance.
4. Generate explainability artifacts:
   - classification report
   - confusion matrix CSV
   - confusion matrix image
   - feature importance CSV
   - feature importance chart
   - top feature summary JSON
5. Infer MLflow model signature.
6. Log metrics, model, signature, input example, and explainability artifacts in the **same MLflow run**.
7. Create `model_uri` from the same active run.
8. Register the model using the same-run `model_uri`.

This ensures that the registered model version points back to the same run that contains the model and its explanation artifacts.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, matplotlib, and JSON utilities.

The key idea is to create explanation files **before MLflow logging** and then log them inside the same model run.


In [ ]:
# Import os to create folders and manage paths.
import os

# Import json to save top feature explanation metadata.
import json

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import datetime to add timestamp to explanation metadata.
from datetime import datetime

# Import MLflow for tracking, model logging, artifacts, and registry.
import mlflow

# Import MLflow sklearn flavor to log sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for tabular data processing.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for explanation charts.
import matplotlib.pyplot as plt

# Import train_test_split for splitting data.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing different column types.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical variables.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical variables.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for model training.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics and confusion matrix.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


## 2. Configure MLflow and Local Folders

This notebook uses local MLflow tracking:

```text
file:./mlruns
```

The explanation artifacts are saved in:

```text
outputs/
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_17_explainability_artifacts"

# Define run name.
RUN_NAME = "17_explainability_artifacts_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for explanation artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for support files.
os.makedirs("artifacts", exist_ok=True)

# Configure local MLflow tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load Dataset

We use the customer churn dataset.

The model predicts whether a customer is likely to churn.


In [ ]:
# Check whether dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise a clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Print dataset shape.
print("Dataset shape:", df.shape)


## 4. Prepare Features, Target, and Preprocessing

We separate:

- input features
- target column
- categorical features
- numerical features

The preprocessing and model are packaged together as a single scikit-learn pipeline.


In [ ]:
# Create feature matrix by removing ID and target columns.
X = df.drop(columns=[id_column, target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical features.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical features.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split dataset into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature details.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 5. Train and Evaluate Model

We train a Random Forest model.

Random Forest is useful for this explainability demo because it provides feature importance values.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=180,
    max_depth=8,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train Random Forest model.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on test data.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report text.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 6. Create Classification Report Artifact

The classification report is useful for explaining precision, recall, and F1-score per class.

We save it as:

```text
outputs/classification_report.txt
```


In [ ]:
# Save classification report as text artifact.
with open("outputs/classification_report.txt", "w") as file:
    file.write(report)

# Print confirmation.
print("Saved: outputs/classification_report.txt")


## 7. Create Confusion Matrix Artifacts

A confusion matrix shows:

- true positives
- true negatives
- false positives
- false negatives

We save both CSV and image versions.


In [ ]:
# Create confusion matrix.
cm = confusion_matrix(y_test, predictions)

# Create confusion matrix dataframe.
confusion_matrix_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"]
)

# Save confusion matrix as CSV.
confusion_matrix_df.to_csv("outputs/confusion_matrix.csv")

# Plot confusion matrix.
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.xticks([0, 1], ["0", "1"])
plt.yticks([0, 1], ["0", "1"])
plt.colorbar()

# Add values inside confusion matrix cells.
for row_index in range(cm.shape[0]):
    for col_index in range(cm.shape[1]):
        plt.text(
            col_index,
            row_index,
            cm[row_index, col_index],
            ha="center",
            va="center"
        )

# Save confusion matrix image.
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png")
plt.close()

# Display confusion matrix dataframe.
display(confusion_matrix_df)

# Print confirmation.
print("Saved: outputs/confusion_matrix.csv")
print("Saved: outputs/confusion_matrix.png")


## 8. Create Feature Importance Artifacts

Feature importance helps explain which transformed features are most influential.

Since the pipeline includes one-hot encoding, we use transformed feature names from the preprocessor.


In [ ]:
# Get transformed feature names from preprocessing step.
transformed_feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

# Get feature importance values from Random Forest model.
feature_importance_values = pipeline.named_steps["model"].feature_importances_

# Create feature importance dataframe.
feature_importance_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "importance": feature_importance_values,
})

# Sort features by importance.
feature_importance_df = feature_importance_df.sort_values("importance", ascending=False)

# Save full feature importance table.
feature_importance_df.to_csv("outputs/feature_importance.csv", index=False)

# Select top 15 important features.
top_features_df = feature_importance_df.head(15)

# Save top features table.
top_features_df.to_csv("outputs/top_15_features.csv", index=False)

# Create top feature importance chart.
plt.figure(figsize=(10, 6))
plt.barh(top_features_df["feature"], top_features_df["importance"])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()

# Save feature importance chart.
plt.savefig("outputs/feature_importance.png")
plt.close()

# Display top features.
display(top_features_df)

# Print confirmation.
print("Saved: outputs/feature_importance.csv")
print("Saved: outputs/top_15_features.csv")
print("Saved: outputs/feature_importance.png")


## 9. Create Explainability Summary JSON

This JSON gives reviewers a compact summary of the explanation artifacts and top features.


In [ ]:
# Create top feature summary list.
top_feature_summary = top_features_df.to_dict(orient="records")

# Create explainability metadata.
explainability_summary = {
    "created_at": datetime.now().isoformat(),
    "model_type": "RandomForestClassifier",
    "explanation_methods": [
        "classification_report",
        "confusion_matrix",
        "feature_importance"
    ],
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    },
    "top_features": top_feature_summary,
    "artifacts": [
        "classification_report.txt",
        "confusion_matrix.csv",
        "confusion_matrix.png",
        "feature_importance.csv",
        "top_15_features.csv",
        "feature_importance.png",
    ],
}

# Save explainability summary as JSON.
with open("outputs/explainability_summary.json", "w") as file:
    json.dump(explainability_summary, file, indent=4)

# Print confirmation.
print("Saved: outputs/explainability_summary.json")


## 10. Infer Model Signature

The model signature captures the expected input/output schema.

This will be logged along with the model artifact.


In [ ]:
# Create input example from test data.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 11. Same-Run MLflow Logging with Explainability Artifacts

This is the corrected and expected implementation.

Inside the **same MLflow run**, we log:

- parameters
- metrics
- tags
- classification report
- confusion matrix CSV
- confusion matrix image
- feature importance CSV
- feature importance image
- explainability summary JSON
- model with signature
- input example
- model URI

Then the model is registered using the same `model_uri`.


In [ ]:
# Start the SAME MLflow run for metrics, explainability artifacts, model, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log explanation methods used.
    mlflow.log_param("explanation_methods", "classification_report,confusion_matrix,feature_importance")

    # Log number of transformed features.
    mlflow.log_param("transformed_feature_count", len(transformed_feature_names))

    # Log number of top features saved.
    mlflow.log_param("top_feature_count", len(top_features_df))

    # Set run purpose tag.
    mlflow.set_tag("run_purpose", "explainability_artifacts")

    # Set explainability availability tag.
    mlflow.set_tag("explainability_artifacts_logged", "true")

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Log top feature importance as metric for quick inspection.
    mlflow.log_metric("top_feature_importance", float(top_features_df.iloc[0]["importance"]))

    # Log classification report artifact.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log confusion matrix CSV artifact.
    mlflow.log_artifact("outputs/confusion_matrix.csv")

    # Log confusion matrix image artifact.
    mlflow.log_artifact("outputs/confusion_matrix.png")

    # Log full feature importance CSV artifact.
    mlflow.log_artifact("outputs/feature_importance.csv")

    # Log top features CSV artifact.
    mlflow.log_artifact("outputs/top_15_features.csv")

    # Log feature importance image artifact.
    mlflow.log_artifact("outputs/feature_importance.png")

    # Log explainability summary JSON artifact.
    mlflow.log_artifact("outputs/explainability_summary.json")

    # Log model WITH input example and signature in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print final model and registry details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 12. Verify Explanation Artifacts

This section verifies that all expected artifacts were created locally.


In [ ]:
# Define expected explainability artifacts.
expected_artifacts = [
    "outputs/classification_report.txt",
    "outputs/confusion_matrix.csv",
    "outputs/confusion_matrix.png",
    "outputs/feature_importance.csv",
    "outputs/top_15_features.csv",
    "outputs/feature_importance.png",
    "outputs/explainability_summary.json",
]

# Verify artifact existence.
for artifact_path in expected_artifacts:
    print(artifact_path, "->", os.path.exists(artifact_path))




- Metrics alone are not enough for model review.
- Explainability artifacts help reviewers understand model behavior.
- Confusion matrix explains error patterns.
- Feature importance explains influential features.
- Classification report gives class-wise precision, recall, and F1-score.
- Artifacts should be logged in the same run as the model.
- The registered model version should point to the run that contains both the model and explanation artifacts.
- This improves auditability, governance, debugging, and stakeholder trust.
